In [13]:
#!/usr/bin/env python3

import json
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from lifelines.utils import concordance_index
from sklearn.model_selection import StratifiedKFold, KFold

from extraction_tools import extract_features



INPUT_DIR = Path("../../data/task3/agent_input")

CLINICAL_FILE = (
    "prostate-time-to-recurrence-or-last-follow-up-clinical-data.json"
)

OUTCOME_FILE = (
    "prostate-time-to-recurrence-or-last-follow-up.json"
)

OUTPUT_DIR = Path("artifacts_task3")
OUTPUT_DIR.mkdir(exist_ok=True)

FEATURES = [
    "bx_isup_max",
    # "bx_missing",
    "rp_isup",
    "pt_stage_num",
    "margin_positive",
    "node_positive",
    "lvi",
]


def concordance_index_gc(
    times: list[float], preds: list[float], events: list[int]
) -> float | None:
    """Harrell's concordance index.

    `preds` are predicted months-to-recurrence used as a risk ordering: a
    shorter predicted time means higher predicted risk. A pair is comparable
    when the earlier subject had an observed event (event=1).
    """
    num = 0.0
    den = 0.0
    n = len(times)
    for i in range(n):
        if events[i] != 1:
            continue
        for j in range(n):
            if i == j or not times[i] < times[j]:
                continue
            den += 1.0
            if preds[i] < preds[j]:
                num += 1.0
            elif preds[i] == preds[j]:
                num += 0.5
    return (num / den) if den > 0 else None


###############################################################################
# LOAD
###############################################################################

rows = []

for case_dir in sorted(INPUT_DIR.iterdir()):

    if not case_dir.is_dir():
        continue

    clinical_path = case_dir / CLINICAL_FILE
    outcome_path = case_dir / OUTCOME_FILE

    if not clinical_path.exists():
        continue

    if not outcome_path.exists():
        continue

    with open(clinical_path) as f:
        clinical_data = json.load(f)

    with open(outcome_path) as f:
        outcome_data = json.load(f)

    row = {
        "case_id": case_dir.name,
        **extract_features(clinical_data),
        "time": outcome_data["months_to_recurrence"],
        "event": outcome_data["event"],
    }

    rows.append(row)

df = pd.DataFrame(rows)

print("Loaded:", len(df))


###############################################################################
# IMPUTATION
###############################################################################

feature_medians = {}

# for feature in FEATURES:

#     median = float(df[feature].median())

#     feature_medians[feature] = median

#     df[feature] = df[feature].fillna(median)

with open(
    OUTPUT_DIR / "feature_medians.json",
    "w",
) as f:
    json.dump(feature_medians, f, indent=2)

with open(
    OUTPUT_DIR / "feature_list.json",
    "w",
) as f:
    json.dump(FEATURES, f, indent=2)


###############################################################################
# TARGET
###############################################################################

cox_target = np.where(
    df["event"] == 1,
    df["time"],
    -df["time"],
)

X = df[FEATURES]

times = df["time"].values
events = df["event"].values


###############################################################################
# CV
###############################################################################

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)


# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42,
# )

oof_risk = np.zeros(len(df))
fold_assignment = np.full(len(df), -1)

feature_importances = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X, events)
):

    print()
    print(f"Fold {fold}")

    fold_assignment[valid_idx] = fold

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = cox_target[train_idx]

    model = CatBoostRegressor(
        loss_function="Cox",
        iterations=1000,
        learning_rate=0.03,
        depth=4,
        verbose=False,
        random_seed=42,
    )

    model.fit(X_train, y_train)

    model.save_model(
        OUTPUT_DIR / f"fold_{fold}.cbm"
    )

    oof_risk[valid_idx] = model.predict(X_valid)
    
    risk = model.predict(X_valid)

    oof_risk[valid_idx] = risk

    fold_cindex = concordance_index(
        event_times=times[valid_idx],
        predicted_scores=-risk,
        event_observed=events[valid_idx],
    )

    print(
        f"Fold {fold}: "
        f"C-index={fold_cindex:.4f}"
    )

    fi = pd.DataFrame(
        {
            "feature": FEATURES,
            "importance": model.get_feature_importance(),
            "fold": fold,
        }
    )

    feature_importances.append(fi)



###############################################################################
# Calculating Months to recurrence from risk score
###############################################################################

RISK_MIN = float(oof_risk.min())
RISK_MAX = float(oof_risk.max())

# I'd use recurrence times only
event_times = times[events == 1]

TIME_MIN = float(event_times.min())
TIME_MAX = float(event_times.max())


def risk_to_months(risk):

    mid = (RISK_MIN + RISK_MAX) / 2
    scale = (RISK_MAX - RISK_MIN) / 6

    p = 1.0 / (
        1.0 + np.exp(
            (risk - mid) / scale
        )
    )

    return (
        TIME_MIN
        + p * (TIME_MAX - TIME_MIN)
    )


oof_months = np.array(
    [
        risk_to_months(r)
        for r in oof_risk
    ]
)


###############################################################################
# OOF METRIC
###############################################################################

c_index = concordance_index(
    event_times=times,
    predicted_scores=-oof_risk,
    event_observed=events,
)

gc_cindex = concordance_index_gc(
    times.tolist(),
    oof_months.tolist(),
    events.tolist(),
)

print()
print("=" * 80)
print("OOF SURVIVAL PERFORMANCE")
print("=" * 80)

print(
    "Cox C-index:",
    round(c_index, 4)
)

print(
    "GC Concordance:",
    round(gc_cindex, 4)
)


###############################################################################
# Save optimal threshold
###############################################################################

best_threshold = None
best_score = -1

for threshold in np.linspace(
    oof_risk.min(),
    oof_risk.max(),
    500,
):

    pred_event = (
        oof_risk >= threshold
    ).astype(int)

    score = (
        pred_event == events
    ).mean()

    if score > best_score:
        best_score = score
        best_threshold = threshold

print(
    "Best threshold:",
    best_threshold
)

print(
    "Best accuracy:",
    best_score,
)

###############################################################################
# SAVE FOLDS
###############################################################################

fold_df = pd.DataFrame(
    {
        "case_id": df["case_id"],
        "fold": fold_assignment,
        "event": df["event"],
        "time": df["time"],
    }
)

fold_df.to_csv(
    OUTPUT_DIR / "folds.csv",
    index=False,
)

case_to_fold = {
    row.case_id: int(row.fold)
    for row in fold_df.itertuples()
}

with open(
    OUTPUT_DIR / "fold_assignments.json",
    "w",
) as f:
    json.dump(
        case_to_fold,
        f,
        indent=2,
    )

print()
print("Saved:", OUTPUT_DIR / "folds.csv")


###############################################################################
# SAVE OOF
###############################################################################

oof_df = pd.DataFrame(
    {
        "case_id": df["case_id"],
        "fold": fold_assignment,
        "event": df["event"],
        "time": df["time"],
        "risk_score": oof_risk,
    }
)

oof_df.to_csv(
    OUTPUT_DIR / "oof_predictions.csv",
    index=False,
)

print()
print("Saved:", OUTPUT_DIR / "oof_predictions.csv")


###############################################################################
# FEATURE IMPORTANCE
###############################################################################

fi_df = pd.concat(
    feature_importances,
    ignore_index=True,
)

mean_fi = (
    fi_df.groupby("feature")["importance"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

mean_fi.to_csv(
    OUTPUT_DIR / "feature_importance.csv",
    index=False,
)

print()
print(mean_fi)


###############################################################################
# FULL MODEL
###############################################################################

full_model = CatBoostRegressor(
    loss_function="Cox",
    iterations=1000,
    learning_rate=0.03,
    depth=4,
    verbose=False,
    random_seed=42,
)

full_model.fit(
    X,
    cox_target,
)

full_model.save_model(
    OUTPUT_DIR / "model_full.cbm"
)

print()
print("Saved:", OUTPUT_DIR / "model_full.cbm")

metadata = {
    "oof_c_index": float(c_index),

    "risk_min": float(oof_risk.min()),
    "risk_max": float(oof_risk.max()),

    "time_min": float(event_times.min()),
    "time_max": float(event_times.max()),
    
    "event_threshold": float(best_threshold)
}



with open(
    OUTPUT_DIR / "metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

Loaded: 75

Fold 0
Fold 0: C-index=0.7400

Fold 1
Fold 1: C-index=0.7692

Fold 2
Fold 2: C-index=0.6296

Fold 3
Fold 3: C-index=0.9400

Fold 4
Fold 4: C-index=0.8421

OOF SURVIVAL PERFORMANCE
Cox C-index: 0.7965
GC Concordance: 0.7965
Best threshold: 0.7257912210151263
Best accuracy: 0.8

Saved: artifacts_task3/folds.csv

Saved: artifacts_task3/oof_predictions.csv

           feature  importance
0      bx_isup_max   30.788138
1     pt_stage_num   19.199249
2  margin_positive   16.170175
3    node_positive   14.452162
4              lvi   10.177075
5          rp_isup    9.213201

Saved: artifacts_task3/model_full.cbm
